# Test Run DINO with FracNet

In [1]:
# activated dependecies
import torch
import os
import torchvision
import pandas as pd
from torchvision.transforms import v2
from transformers.image_utils import load_image

/home/finn/Documents/1-projects/Consulting-Project-simple-Prediction-Model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# set reference to DINO repo and weights
REPO_DIR = "dinov3"
CHECKPOINT_PATH = "weights/dinov3_vits16_pretrain_lvd1689m-08c60483.pth"

# initiate pretrained DINOv3 ViT model 
dinov3_vits16 = torch.hub.load(REPO_DIR, 'dinov3_vits16', source='local', weights=CHECKPOINT_PATH)

In [3]:
def slice_ct(ct_tensor: torch.Tensor, slice_index: int):
    if ct_tensor.ndim != 5 or ct_tensor.shape[0] != 1 or ct_tensor.shape[1] != 1:
        raise ValueError(f"Expected shape [1, 1, D, H, W], got {tuple(ct_tensor.shape)}")
    
    depth = ct_tensor.shape[2]
    if not (0 <= slice_index < depth):
        raise ValueError(f"slice_index must be between 0 and {depth-1}, got {slice_index}")
    
    return ct_tensor[0, 0, slice_index, :, :].cpu().numpy()

def run_dino_on_single_img(image):

    # apllying transformation function (make tensor --> resize --> make float --> normalize)
    img_latents = make_transform()(ct_slice).unsqueeze(0)

    with torch.inference_mode():
        feats = dinov3_vits16.forward_features(img_latents) 

    if isinstance(feats, dict):
        pooled_output = feats["x_norm_clstoken"]
        patch_tokens = feats["x_norm_patchtokens"]
    else:
        # fallback if forward_features() not implemented
        tokens = model(img_latents) 
        pooled_output = tokens[:,0]          
        patch_tokens = tokens[:,1+model.num_register_tokens:,:]

    latents = [pooled_output, patch_tokens]
    return latents

def get_all_ct_latents(ct_image):
    global_latents_list = []
    local_latents_list = []

    for slice_index in range(ct_image.shape[2]):
        # extract slice
        ct_slice = slice_ct(ct_img, slice_index=slice_index)

        # create latents
        global_latents, local_latents = run_dino_on_single_img(ct_slice)
        
        # append latents of each step to one list
        global_latents_list.append(global_latents)
        local_latents_list.append(local_latents)

    # stack python list to a one tensor
    global_latents_stacked = torch.stack(global_latents_list)
    local_latents_stacked = torch.stack(local_latents_list)

    return [global_latents_stacked, local_latents_stacked]


In [4]:
def make_transform(resize_size: int = 256):
    to_tensor = v2.ToImage()
    resize = v2.Resize((resize_size, resize_size), antialias=True)
    to_float = v2.ToDtype(torch.float32, scale=True)
    normalize = v2.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return v2.Compose([to_tensor, resize, to_float, normalize])